# Week 7, Day 2: Low-rank approximation: keeping the structure, dropping the noise

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w007/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The best rank-k approximation

Write `rank_k(A, k)` that returns the best rank-k approximation of `A` from its economy SVD. Then check the error formula for one example, and compare with a random rank-k matrix, which should do much worse.

In [ ]:
import numpy as np
def rank_k(A, k):
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    return ...

rng = np.random.default_rng(0)
A = rng.normal(size=(12, 8))
s = np.linalg.svd(A, compute_uv=False)
print("error:", np.linalg.norm(A - rank_k(A, 3)).round(4), "  formula:", np.sqrt(np.sum(s[3:] ** 2)).round(4))
R3 = rng.normal(size=(12, 3)) @ rng.normal(size=(3, 8)) * 0.3
print("a random rank-3 matrix:", np.linalg.norm(A - R3).round(4))

In [ ]:
import numpy as np
def test_rank():
    "rank_k returns a matrix of the right shape and rank"
    M = np.random.default_rng(1).normal(size=(8, 6))
    Mk = rank_k(M, 2)
    assert Mk.shape == (8, 6) and np.linalg.matrix_rank(Mk) == 2

def test_error():
    "the Frobenius error equals the root sum of the discarded squared singular values"
    M = np.random.default_rng(2).normal(size=(10, 7))
    s = np.linalg.svd(M, compute_uv=False)
    for k in [1, 3, 6]:
        assert np.isclose(np.linalg.norm(M - rank_k(M, k)), np.sqrt(np.sum(s[k:] ** 2)))

def test_full():
    "keeping every component gives back the matrix"
    M = np.random.default_rng(3).normal(size=(5, 5))
    assert np.allclose(rank_k(M, 5), M)

test_rank()
test_error()
test_full()
print("All checks passed")

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(36)
N, T = 40, 60
t = np.linspace(0, 1, T)
latents = np.array([np.sin(np.pi * t), np.sin(2 * np.pi * t), np.exp(-(t - 0.7) ** 2 / 0.02)])   # (3, 60)
loadings = rng.normal(size=(N, 3)) * [1, 0.8, 0.8]                                               # (40, 3)
truth = loadings @ latents                     # rank 3
R = truth + 0.4 * rng.normal(size=(N, T))      # what we record
s_R = np.linalg.svd(R, compute_uv=False)
plt.figure(figsize=(5, 3)); plt.plot(s_R, "o"); plt.xlabel("component"); plt.ylabel("singular value"); plt.show()
print("top singular values:", s_R[:6].round(2))

### Exercise: How many components should you keep?

For k from 1 to 20, compute the root-mean-square error between `rank_k(R, k)` and the true signal `truth`, storing the 20 values in `errs`. Store the best k in `best_k` and the raw recording's error in `raw_err`. Plot `errs` against k with `raw_err` as a horizontal line.

In [ ]:
ks = np.arange(1, 21)
errs = [...]
best_k = ...
raw_err = ...

plt.figure(figsize=(5, 3)); plt.plot(ks, errs, "o-"); plt.axhline(raw_err, ls="--", color="0.5", label="raw recording")
plt.xlabel("components kept, k"); plt.ylabel("RMS error to truth"); plt.legend(frameon=False); plt.show()
print("best k:", best_k, " error", round(min(errs), 3), " raw", round(raw_err, 3))

In [ ]:
import numpy as np
def test_errs():
    "errs holds the RMS error to the truth for k = 1 to 20"
    assert len(errs) == 20
    assert np.isclose(errs[2], np.sqrt(np.mean((rank_k(R, 3) - truth) ** 2)))

def test_best():
    "best_k is the k with the smallest error, and it beats the raw recording"
    assert best_k == int(np.argmin(errs)) + 1
    assert best_k == 3
    assert min(errs) < raw_err / 2

test_errs()
test_best()
print("All checks passed")

### Exercise: Is the receptive field separable?

Write `separability(M)`. Compute it for both receptive fields, store the results in `sep_simple` and `sep_ds`, and store the best rank-one approximation of the simple cell's receptive field in `rf1`. Plot all three.

In [ ]:
x = np.linspace(-2, 2, 20)
lag = np.arange(15)
X, L = np.meshgrid(x, lag, indexing="ij")
temporal = L * np.exp(-L / 3) * np.sin(L / 2)
rf_simple = np.exp(-X ** 2 / 0.5) * np.cos(3 * X) * temporal + 0.02 * rng.normal(size=X.shape)
rf_ds = np.exp(-X ** 2 / 0.8) * np.cos(3 * (X - 0.25 * L)) * L * np.exp(-L / 3)     # tilted in space-time

def separability(M):
    s = np.linalg.svd(M, compute_uv=False)
    return ...

sep_simple, sep_ds = ..., ...
rf1 = ...

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, M, title in zip(ax, [rf_simple, rf1, rf_ds], ["simple cell", "its rank-1 fit", "direction selective"]):
    a.imshow(M, cmap="RdBu_r", aspect="auto"); a.set_title(title); a.set_xlabel("time lag"); a.set_ylabel("position")
plt.tight_layout(); plt.show()
print("separability: simple", round(sep_simple, 3), " direction selective", round(sep_ds, 3))

In [ ]:
import numpy as np
def test_index():
    "separability returns sigma_1 squared over the sum of squared singular values"
    M = np.outer([1.0, 2.0, 3.0], [1.0, -1.0])
    assert np.isclose(separability(M), 1.0)
    assert np.isclose(separability(np.eye(4)), 0.25)

def test_cells():
    "the simple cell is close to separable, the direction-selective cell is not"
    assert sep_simple > 0.95 and sep_ds < 0.7

def test_fit():
    "rf1 is the best rank-one approximation of the simple cell's receptive field"
    assert np.allclose(rf1, rank_k(rf_simple, 1))

test_index()
test_cells()
test_fit()
print("All checks passed")